**BBM473 Database Laboratory (Spring 2025)**


Exercise 2: More on SQL
-----

Please run the below cells first before proceeding- you'll need them soon!

In [391]:
%load_ext sql
%sql sqlite://

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [392]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [393]:
%%sql
DROP TABLE IF EXISTS Movies;
CREATE TABLE Movies(title VARCHAR(50), year INT, director VARCHAR(50), length INT);
INSERT INTO Movies VALUES('Database Wars', 1967, 'John Joe', 123);
INSERT INTO Movies VALUES('The Databaser', 1992, 'John Bob', 190);
INSERT INTO Movies VALUES('Database Wars', 1998, 'John Jim', 176);

 * sqlite://
   sqlite:///dataset_1.db
Done.
Done.
1 rows affected.
1 rows affected.
1 rows affected.


[]

In [394]:
%sql select * from Movies

 * sqlite://
   sqlite:///dataset_1.db
Done.


title,year,director,length
Database Wars,1967,John Joe,123
The Databaser,1992,John Bob,190
Database Wars,1998,John Jim,176


In [395]:
%sql DROP TABLE IF EXISTS A; DROP TABLE IF EXISTS B;
%sql CREATE TABLE A (x int, y int); CREATE TABLE B (x int, y int);
for i in range(1,6):
    %sql INSERT INTO A VALUES (:i, :i+1)
for i in range(1,11,3):
    %sql INSERT INTO B VALUES (:i, :i+2)

 * sqlite://
   sqlite:///dataset_1.db
Done.
Done.
 * sqlite://
   sqlite:///dataset_1.db
Done.
Done.
 * sqlite://
   sqlite:///dataset_1.db
1 rows affected.
 * sqlite://
   sqlite:///dataset_1.db
1 rows affected.
 * sqlite://
   sqlite:///dataset_1.db
1 rows affected.
 * sqlite://
   sqlite:///dataset_1.db
1 rows affected.
 * sqlite://
   sqlite:///dataset_1.db
1 rows affected.
 * sqlite://
   sqlite:///dataset_1.db
1 rows affected.
 * sqlite://
   sqlite:///dataset_1.db
1 rows affected.
 * sqlite://
   sqlite:///dataset_1.db
1 rows affected.
 * sqlite://
   sqlite:///dataset_1.db
1 rows affected.


In [396]:
%sql select * from A

 * sqlite://
   sqlite:///dataset_1.db
Done.


x,y
1,2
2,3
3,4
4,5
5,6


In [397]:
%sql select * from B

 * sqlite://
   sqlite:///dataset_1.db
Done.


x,y
1,3
4,6
7,9
10,12


Activity 2-1: (30pts.)
------------

ORDER BY semantics, set operators & nested queries

In [398]:
%sql SELECT * FROM movies

 * sqlite://
   sqlite:///dataset_1.db
Done.


title,year,director,length
Database Wars,1967,John Joe,123
The Databaser,1992,John Bob,190
Database Wars,1998,John Jim,176


Task #1 (10 pts.)
-----------

**Can you write the movie query below as a single SFW query?**

Recall that we are trying to find **all movie titles that were used for more than one movie.** You may assume that no two movies in the same year have the same title. Our schema for the `movies` table is:

> * title STRING
> * year INT
> * director STRING
> * length INT

Let's try to write the nested query that solves this:

In [399]:
%%sql
SELECT *
FROM Movies m
WHERE (SELECT count(*) FROM Movies WHERE title = m.title) > 1;

 * sqlite://
   sqlite:///dataset_1.db
Done.


title,year,director,length
Database Wars,1967,John Joe,123
Database Wars,1998,John Jim,176


In [400]:
%%sql
SELECT distinct m.title
FROM Movies m
WHERE (SELECT count(*) FROM Movies WHERE title = m.title) > 1;

 * sqlite://
   sqlite:///dataset_1.db
Done.


title
Database Wars


Now write another SFW query to solve the problem:

In [ ]:
%%sql
SELECT Distinct m1.title
FROM Movies m1, Movies m2
WHERE m1.title = m2.title And (m1.year <> m2.year);

 * sqlite://
   sqlite:///dataset_1.db
Done.


title
Database Wars


Task #2 (20 pts.)
--------------------

Consider the two relations $A$ and $B$ below:

In [403]:
%sql SELECT * FROM A;

 * sqlite://
   sqlite:///dataset_1.db
Done.


x,y
1,2
2,3
3,4
4,5
5,6


In [404]:
%sql SELECT * FROM B;

 * sqlite://
   sqlite:///dataset_1.db
Done.


x,y
1,3
4,6
7,9
10,12


Assuming no duplicates, can you write an `INTERSECT` query, **just over the $x$ attribute**, without using `INTERSECT` OR nested queries?  Write your query here:

In [405]:
%%sql
Select A.x
From A Join B 
On A.x = B.x;

 * sqlite://
   sqlite:///dataset_1.db
Done.


x
1
4


What is this operation called? 

-It is called inner join or equijoin

Next, using set operators again as well, can you return all the _full_ tuples in $A$ and $B$ that overlap in $x$ attributes?  Write your query here:

In [406]:
%%sql
Select A.*
From A Join B 
On A.x = B.x
Union
Select B.*
From B Join A 
On B.x=A.x;

 * sqlite://
   sqlite:///dataset_1.db
Done.


x,y
1,2
1,3
4,5
4,6


Execute the code below to connect to the database:

In [407]:
%sql sqlite:///dataset_1.db

Activity 2-2 (30 pts.)
------------
Aggregation operators, GROUP BY

Task #1 (20 pts.)
-----------

Consider a set of tables that describe the up-and-coming bagel startup industry; for now let's just look at two tables here, `bagel`, which describes types of bagels made by the different bagel companies:
> * name STRING
> * price FLOAT
> * made_by STRING

And `purchase`:
> * bagel_name STRING
> * franchise STRING
> * date INT
> * quantity INT
> * purchaser_age INT

Where `purchase.bagel_name` references `bagel.name` and `purchase.franchise` references `bagel.made_by`:

In [408]:
%sql SELECT * FROM bagel LIMIT 3;

   sqlite://
 * sqlite:///dataset_1.db
Done.


name,price,made_by
Plain with shmear,1.99,Bobs Bagels
Egg with shmear,2.39,Bobs Bagels
eBagel Drinkable Bagel,27.99,eBagel


In [409]:
%sql SELECT * FROM purchase LIMIT 3;

   sqlite://
 * sqlite:///dataset_1.db
Done.


bagel_name,franchise,date,quantity,purchaser_age
Plain with shmear,Bobs Bagels,1,12,28
Egg with shmear,Bobs Bagels,2,6,47
Plain with shmear,BAGEL CORP,2,12,24


Can you write a query to get the _total revenue_ for each bagel type **which had an average purchaser age over 18**?  Type your query below:

In [410]:
%%sql 
Select p.bagel_name, Sum(price*quantity) As total_revenue
From bagel As b Join purchase As p
On p.bagel_name = b.name And p.franchise = b.made_by
Group By p.bagel_name
Having Avg(p.purchaser_age) > 18;

   sqlite://
 * sqlite:///dataset_1.db
Done.


bagel_name,total_revenue
Egg with shmear,14.34
Plain with shmear,84.50999999999999


Task #2 (10 pts.)
-----------

Here we'll use a simplified version of the `precipitation_full` table, which just has _daily_ rainfall _in CA only_, and has the following schema:

> * station_id
> * day
> * precipitation

In [411]:
%sql SELECT * FROM precipitation LIMIT 5;

   sqlite://
 * sqlite:///dataset_1.db
Done.


station_id,day,precipitation
16102,1,10
16102,4,10
16102,24,30
21201,1,0
21201,20,10


We want to get station_ids which have average precipitations > 75.  Try doing this first as a nested query:

In [412]:
%%sql
SELECT Distinct p.station_id
FROM precipitation p
WHERE (SELECT AVG(precipitation) FROM precipitation p1 WHERE p.station_id = p1.station_id) > 75;

   sqlite://
 * sqlite:///dataset_1.db
Done.


station_id
88302
250002
335701
357302
488301


Now, try re-writing as a GROUP BY:

In [413]:
%%sql
Select station_id
From precipitation
Group By station_id
Having Avg(precipitation) > 75

   sqlite://
 * sqlite:///dataset_1.db
Done.


station_id
88302
250002
335701
357302
488301


In [414]:
%%sql
Select station_id, Avg(precipitation)
From precipitation
Group By station_id
Having Avg(precipitation) > 75

   sqlite://
 * sqlite:///dataset_1.db
Done.


station_id,Avg(precipitation)
88302,95.0
250002,92.5
335701,83.63636363636364
357302,83.33333333333333
488301,75.71428571428571


Now time it by using `%time` followed by single-line versions of your queries above (clunky, but will work) to see how they compare!

**Note:** Yes, currently the answers are filled in below for convenience... but you should still try getting them on your own above!

In [415]:
%%time
%%sql
SELECT DISTINCT p.station_id
FROM precipitation p
WHERE (SELECT AVG(precipitation) FROM precipitation WHERE station_id = p.station_id) > 75;

   sqlite://
 * sqlite:///dataset_1.db
Done.
CPU times: user 15.2 ms, sys: 1.22 ms, total: 16.4 ms
Wall time: 15.6 ms


In [416]:
%time %sql SELECT p.station_id FROM precipitation p GROUP BY p.station_id HAVING AVG(p.precipitation) > 75;

   sqlite://
 * sqlite:///dataset_1.db
Done.
CPU times: user 1.78 ms, sys: 1.02 ms, total: 2.8 ms
Wall time: 1.95 ms


**An ~ 10-20x difference in execution time!!**

Activity 2-3 (40 pts.)
------------
Quantifiers, NULLs, and Outer Joins

Task #1 (40 pts.)
-----------

Recall that the tables we just looked at:

`bagel`, which describes types of bagels made by the different bagel companies:
> * name STRING
> * price FLOAT
> * made_by STRING

And `purchase`:
> * bagel_name STRING
> * franchise STRING
> * date INT
> * quantity INT
> * purchaser_age INT

Where `purchase.bagel_name` references `bagel.name` and `purchase.franchise` references `bagel.made_by`.

**Can you find out if there were any purchases of products not on one of the company's official lists (i.e. the `bagel` table), using a single SQL query?**

Write your query here:

In [417]:
%%sql
Select * From bagel Limit 3

   sqlite://
 * sqlite:///dataset_1.db
Done.


name,price,made_by
Plain with shmear,1.99,Bobs Bagels
Egg with shmear,2.39,Bobs Bagels
eBagel Drinkable Bagel,27.99,eBagel


In [418]:
%%sql
Select * From purchase Limit 3

   sqlite://
 * sqlite:///dataset_1.db
Done.


bagel_name,franchise,date,quantity,purchaser_age
Plain with shmear,Bobs Bagels,1,12,28
Egg with shmear,Bobs Bagels,2,6,47
Plain with shmear,BAGEL CORP,2,12,24


In [419]:
%%sql
Select *
From purchase As p
Left Outer Join bagel As b 
On p.bagel_name = b.name AND p.franchise = b.made_by
Where b.name IS NULL;


   sqlite://
 * sqlite:///dataset_1.db
Done.


bagel_name,franchise,date,quantity,purchaser_age,name,price,made_by
Moonshine,BAGEL CORP,7,1000,37,None,None,None


Save and submit your notebook file (.ipynb) to the form below with your name and student ID. Do not include the database file or any other file. Only one notebook file is sufficient.

https://forms.gle/fov1qy6dp4txxF3H9

**Note:** *You cannot submit your exercise notebook remotely. You have to be at the class.*